# Figures for the README, drawn from `outputs/*.csv` only

Run the analysis notebooks first (see *How to run* in the README). This notebook reads result CSVs from `outputs/` and writes PNGs to `images/`.
No result number is typed in here; every number in a figure, label or caption comes from a CSV.

- `TRANSPARENT = False`: white background, saved to `images/`.
- `TRANSPARENT = True`: transparent background for the portfolio site, saved to `images/transparent/`.

In [1]:
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker
from matplotlib import font_manager
from matplotlib.colors import to_rgb
from matplotlib.patches import Rectangle

from config import OUT_DIR, FIG_DIR, CAMP_MIN_WEEK, CAMP_MAX_WEEK

TRANSPARENT = False   # True: transparent PNGs for the site -> images/transparent/
DPI = 200
SAVE_DIR = FIG_DIR / "transparent" if TRANSPARENT else FIG_DIR
SAVE_DIR.mkdir(parents=True, exist_ok=True)

# palette (CLAUDE.md)
INK, INK2, INK3 = "#23233A", "#4A4A66", "#7B7B96"
VIOLET, ROSE, BLUE, MINT = "#7B6FD8", "#D9536B", "#4C7FE0", "#2F9E88"
GRID = "#E6E6EF"

def tint(color, a):
    """Blend a colour toward white; a=1 keeps the colour, a=0 is white."""
    return tuple(1 - (1 - np.array(to_rgb(color))) * a)

_available = {f.name for f in font_manager.fontManager.ttflist}
FONT = "IBM Plex Sans" if "IBM Plex Sans" in _available else "DejaVu Sans"
plt.rcParams.update({
    "font.family": FONT, "font.size": 10,
    "text.color": INK, "axes.labelcolor": INK2, "xtick.color": INK3, "ytick.color": INK3,
    "axes.edgecolor": GRID, "axes.linewidth": 1.0,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.facecolor": "none" if TRANSPARENT else "white",
    "figure.facecolor": "none" if TRANSPARENT else "white",
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.8, "axes.axisbelow": True,
    "axes.unicode_minus": True,
})
print("font:", FONT, "| transparent:", TRANSPARENT, "| save dir:", SAVE_DIR.name)

def read(name):
    return pd.read_csv(OUT_DIR / f"{name}.csv")

def head(fig, title, subtitle, top=0.80):
    fig.subplots_adjust(top=top)
    fig.canvas.draw()
    r = fig.canvas.get_renderer()
    x0 = min(ax.get_tightbbox(r).x0 for ax in fig.axes)          # left edge of axis labels
    x = fig.transFigure.inverted().transform((x0, 0))[0]
    fig.text(x, 0.995, title, ha="left", va="top", fontsize=14, fontweight="bold", color=INK)
    fig.text(x, 0.995 - 0.075, subtitle, ha="left", va="top", fontsize=9, color=INK3, linespacing=1.5)

def save(fig, name):
    fig.savefig(SAVE_DIR / f"{name}.png", dpi=DPI, bbox_inches="tight", pad_inches=0.25,
                transparent=TRANSPARENT, facecolor="none" if TRANSPARENT else "white")
    plt.close(fig)
    print("saved", SAVE_DIR.name + "/" + name + ".png")

def sgn(v, nd=2):
    return f"{v:+.{nd}f}".replace("-", "\u2212")

def neg(s):
    return s.replace("-", "\u2212")

TIERS = ["low", "mid", "high"]
TIER_LABEL = {"low": "Low", "mid": "Medium", "high": "High"}
TYPE_COLOR = {"TypeA": BLUE, "TypeB": VIOLET, "TypeC": ROSE}

font: DejaVu Sans | transparent: False | save dir: images


## a. `campaign-timeline.png`

In [2]:
tl = read("campaign_timeline").sort_values(["start_week", "campaign"], kind="stable").reset_index(drop=True)
counts = tl["type"].value_counts()
fig, ax = plt.subplots(figsize=(8.6, 7.4))
y = np.arange(len(tl))[::-1]
for yy, r in zip(y, tl.itertuples()):
    col = TYPE_COLOR[r.type]
    strong = r.type == "TypeC"
    s, e = max(r.start_week, CAMP_MIN_WEEK), min(r.end_week, CAMP_MAX_WEEK)
    ax.barh(yy, e - s + 1, left=s - 0.5, height=0.66, color=col if strong else tint(col, 0.40), zorder=3)
    if r.end_week > CAMP_MAX_WEEK:
        ax.annotate("", xy=(CAMP_MAX_WEEK + 2.2, yy), xytext=(CAMP_MAX_WEEK + 0.6, yy),
                    arrowprops=dict(arrowstyle="-|>", color=col, lw=1.4))
ax.set_yticks(y); ax.set_yticklabels([f"#{c}" for c in tl["campaign"]], fontsize=8)
ax.set_xlim(CAMP_MIN_WEEK - 1, CAMP_MAX_WEEK + 4)
ax.set_xticks(range(35, CAMP_MAX_WEEK + 1, 10))
ax.set_xlabel("Week (panel weeks 33\u2013101)")
ax.grid(axis="y", visible=False)
ax.legend(handles=[Rectangle((0, 0), 1, 1, fc=(TYPE_COLOR[t] if t == "TypeC" else tint(TYPE_COLOR[t], 0.40)), label=f"{t.replace('Type', 'Type ')} ({counts[t]} campaigns)")
                   for t in ["TypeA", "TypeB", "TypeC"]], frameon=False, loc="lower left", fontsize=9)
_n_past = int((tl["end_week"] > CAMP_MAX_WEEK).sum())
head(fig, f"{len(tl)} campaigns of three types, launched from week {int(tl['start_week'].min())}",
     f"Each bar runs from a campaign's start week to its end week \u00b7 Type C highlighted \u00b7 arrow = continues after week {CAMP_MAX_WEEK} ({_n_past} campaigns)", top=0.88)
save(fig, "campaign-timeline")

saved images/campaign-timeline.png


## b. `sample-heatmap.png`

In [3]:
cs3 = read("cell_size_3x3").set_index("tier").loc[TIERS]
chk = read("cell_size_checks").iloc[0]
thr = int(chk["cell_size_threshold"])
types = ["TypeA", "TypeB", "TypeC"]
fig, ax = plt.subplots(figsize=(6.6, 4.6))
vmax = cs3.values.max()
for i, tier in enumerate(TIERS[::-1]):            # High on top
    for j, t in enumerate(types):
        v = int(cs3.loc[tier, t]); small = v < thr
        fc = tint(VIOLET, 0.15 + 0.75 * np.log(v) / np.log(vmax))
        ax.add_patch(Rectangle((j, i), 0.96, 0.94, fc=fc, ec=ROSE if small else "none", lw=3 if small else 0, zorder=2))
        ax.text(j + 0.48, i + 0.52, f"{v:,}" + ("\u2020" if small else ""), ha="center", va="center", fontsize=20,
                fontweight="bold", color=ROSE if small else (INK if v < 0.5 * vmax else "white"), zorder=3)
        if small:
            ax.text(j + 0.48, i + 0.2, f"below {thr}:\nreference only", ha="center", va="center", fontsize=8, color=ROSE, zorder=3)
ax.set_xlim(0, 3); ax.set_ylim(0, 3); ax.set_aspect("equal"); ax.axis("off")
for j, t in enumerate(types):
    ax.text(j + 0.48, 3.06, t.replace("Type", "Type "), ha="center", va="bottom", fontsize=11, color=INK2)
for i, tier in enumerate(TIERS[::-1]):
    ax.text(-0.06, i + 0.47, TIER_LABEL[tier], ha="right", va="center", fontsize=11, color=INK2)
head(fig, "Households receiving each campaign type, by spending tier",
     f"Distinct households with the campaign type active at least once, weeks {CAMP_MIN_WEEK}\u2013{CAMP_MAX_WEEK} \u00b7 Type C \u00d7 Low has {int(chk['low_typeC_households'])}", top=0.78)
save(fig, "sample-heatmap")

saved images/sample-heatmap.png


## c. `pretrend.png`

In [4]:
pw = read("pretrend_weekly")
pt = read("pretrend_tests").set_index("scope")
fig, axes = plt.subplots(1, 3, figsize=(12.0, 4.4), sharex=True)
for ax, tier in zip(axes, TIERS):
    s = pw[pw["scope"] == tier]
    n0, n1 = int(pt.loc[tier, "n_never_receive"]), int(pt.loc[tier, "n_will_receive"])
    small = n0 < 100
    for grp, col, lab in [("will_receive", VIOLET, f"Will receive (n = {n1:,})"), ("never_receive", ROSE, f"Never receive (n = {n0:,})")]:
        d = s[s["group"] == grp].sort_values("week_no", kind="stable")
        ax.plot(d["week_no"], d["per_household_spend"], color=col, lw=2.4 if grp == "never_receive" and small else 1.9,
                marker="o", ms=3.5, label=lab, alpha=1.0 if (grp == "will_receive" or small) else 0.75)
    ax.set_title(f"{TIER_LABEL[tier]} spenders", loc="left", fontsize=11, fontweight="bold", color=INK, pad=22)
    ax.text(0, 1.04, f"trend-difference p = {pt.loc[tier, 'p_value']:.3f}" + (f"  \u00b7  only {n0} never-recipients: low power" if small else ""),
            transform=ax.transAxes, fontsize=8.5, color=ROSE if small else INK3, fontweight="bold" if small else "normal")
    ax.legend(frameon=False, fontsize=8.5, loc="upper center", bbox_to_anchor=(0.5, -0.2), ncol=1)
    ax.set_xlabel("Week")
axes[0].set_ylabel("Mean weekly spend per household ($)")
head(fig, "Before any campaign: weekly spending of future recipients and never-recipients",
     f"Weeks {int(pw['week_no'].min())}\u2013{int(pw['week_no'].max())}, tiers fixed on this window \u00b7 p = test that the weekly trend differs between the groups", top=0.74)
save(fig, "pretrend")

saved images/pretrend.png


## d. `forest-plot.png` (overwrites)

In [5]:
fe = read("fe_bh_table")
order = [(t, r) for t in ["TypeA", "TypeB", "TypeC"] for r in TIERS]
fe = fe.set_index(["type", "tier"]).loc[order].reset_index()
n_zero = int(((fe["ci_low"] <= 0) & (fe["ci_high"] >= 0)).sum())
fig, ax = plt.subplots(figsize=(8.4, 5.4))
y = np.arange(len(fe))[::-1]
ax.axvline(0, color=INK, lw=1.4, zorder=2)
for yy, r in zip(y, fe.itertuples()):
    sig = bool(r.reject_bh)
    col = ROSE if sig else tint(INK3, 0.55)
    ax.hlines(yy, r.ci_low, r.ci_high, color=col, lw=3.2 if sig else 2.2, zorder=3)
    ax.plot(r.coef, yy, "o", color=col, ms=10 if sig else 7, zorder=4)
    ax.text(max(r.ci_high, 0) + 0.5, yy, neg(f"{r.coef:.2f}"), va="center", fontsize=9, fontweight="bold" if sig else "normal", color=INK if sig else INK3)
labels = [f"{r.type.replace('Type', 'Type ')} \u00d7 {TIER_LABEL[r.tier]}  (n = {r.n_recipients:,}{chr(0x2020) if r.small_cell else ''})" for r in fe.itertuples()]
ax.set_yticks(y); ax.set_yticklabels(labels)
for lab, r in zip(ax.get_yticklabels(), fe.itertuples()):
    lab.set_color(INK if r.reject_bh else INK3); lab.set_fontweight("bold" if r.reject_bh else "normal")
ax.set_xlabel("Weekly spending difference when the campaign type is active ($ per household-week)")
ax.grid(axis="y", visible=False)
head(fig, "Only Type C \u00d7 High survives the multiple-comparison adjustment",
     f"Two-way fixed effects, household-clustered 95% intervals (not adjusted) \u00b7 {n_zero} of {len(fe)} intervals include zero \u00b7 dark = passes Benjamini\u2013Hochberg \u00b7 \u2020 fewer than 30 households", top=0.85)
save(fig, "forest-plot")

saved images/forest-plot.png


## e. `bh-correction.png` (overwrites)

In [6]:
bh = read("fe_bh_table").sort_values("p_raw", kind="stable").reset_index(drop=True)
fig, ax = plt.subplots(figsize=(8.2, 5.0))
y = np.arange(len(bh))[::-1]
ax.axvline(0.05, color=INK3, ls="--", lw=1.4, zorder=2)
ax.set_ylim(y.min() - 1.0, y.max() + 0.6)
ax.text(0.05 * 0.93, y.min() - 0.62, "0.05", ha="right", va="center", fontsize=9, color=INK3)
for yy, r in zip(y, bh.itertuples()):
    sig = bool(r.reject_bh)
    col = ROSE if sig else tint(INK3, 0.6)
    ax.plot([r.p_raw, r.p_adj_bh], [yy, yy], color=col, lw=2 if sig else 1.4, zorder=3)
    ax.plot(r.p_raw, yy, "o", mfc="white", mec=col, mew=2, ms=8, zorder=4)
    ax.plot(r.p_adj_bh, yy, "D", color=col, ms=8 if sig else 6.5, zorder=5)
    ax.text(max(r.p_raw, r.p_adj_bh) * 1.35, yy, f"{r.p_raw:.4f} \u2192 {r.p_adj_bh:.4f}", va="center", fontsize=8.5,
            fontweight="bold" if sig else "normal", color=INK if sig else INK3)
ax.set_xscale("log")
ax.set_xlim(2e-4, 6)
ax.set_xticks([0.001, 0.01, 0.05, 0.1, 1]); ax.get_xaxis().set_major_formatter(matplotlib.ticker.FuncFormatter(lambda v, _: f"{v:g}"))
ax.xaxis.set_minor_formatter(matplotlib.ticker.NullFormatter())
ax.set_yticks(y); ax.set_yticklabels([f"{r.type.replace('Type', 'Type ')} \u00d7 {TIER_LABEL[r.tier]}" for r in bh.itertuples()])
for lab, r in zip(ax.get_yticklabels(), bh.itertuples()):
    lab.set_color(INK if r.reject_bh else INK3); lab.set_fontweight("bold" if r.reject_bh else "normal")
ax.set_xlabel("p-value (log scale)")
ax.grid(axis="y", visible=False)
_pass = bh[bh["reject_bh"]].iloc[0]
head(fig, "Benjamini\u2013Hochberg adjustment of the nine p-values",
     f"Hollow circle = raw p, filled diamond = BH-adjusted p \u00b7 only Type C \u00d7 High stays below 0.05 (adjusted p = {_pass.p_adj_bh:.4f})", top=0.86)
save(fig, "bh-correction")

saved images/bh-correction.png


## f. `lead-effect.png`

In [7]:
lc = read("lead_effect_coefficients")
ls = read("lead_effect_summary").iloc[0]
pts = [(lc.iloc[1], "5\u20138 weeks\nbefore first\nType C week"), (lc.iloc[0], "1\u20134 weeks\nbefore first\nType C week"),
       (lc.iloc[2], "Type C weeks\n(lead weeks\nincluded)")]
fig, ax = plt.subplots(figsize=(7.2, 5.0))
ax.axhline(0, color=INK, lw=1.4, zorder=2)
for x, (r, lab) in enumerate(pts):
    sig = not (r.ci_low <= 0 <= r.ci_high)
    col = ROSE if sig else tint(INK3, 0.6)
    ax.vlines(x, r.ci_low, r.ci_high, color=col, lw=3.4 if sig else 2.4, zorder=3)
    ax.plot(x, r.coef, "o", color=col, ms=12 if sig else 9, zorder=4)
    ax.text(x + 0.12, r.coef, neg(f"{r.coef:.2f}"), va="center", fontsize=11, fontweight="bold" if sig else "normal", color=INK if sig else INK3)
    ax.text(x, r.ci_low - 0.9, neg(f"[{r.ci_low:.2f}, {r.ci_high:.2f}]"), ha="center", va="top", fontsize=8.5, color=INK3)
ax.set_xticks(range(3)); ax.set_xticklabels([l for _, l in pts], fontsize=9.5)
ax.set_xlim(-0.6, 2.7)
ax.set_ylim(min(p.ci_low for p, _ in pts) - 4, max(p.ci_high for p, _ in pts) + 2.5)
ax.set_ylabel("Weekly spending vs the household's other weeks ($)")
ax.grid(axis="x", visible=False)
ax.text(2.68, ax.get_ylim()[1] - 0.6, f"Type A/B active in {ls['pct_lead4_household_weeks_with_typeAB']:.0f}%\nof the 1\u20134-week lead household-weeks",
        ha="right", va="top", fontsize=8.5, color=INK2)
head(fig, "Spending was already lower in the four weeks before the first Type C week",
     f"Type C \u00d7 High households (n = {int(ls['n_typeC_high_households'])}), main regression plus lead indicators \u00b7 95% intervals, household-clustered \u00b7 dark = interval excludes zero", top=0.85)
save(fig, "lead-effect")

saved images/lead-effect.png


## g. `reach-rate.png` (overwrites)

In [8]:
r3 = read("reach_by_tier3").set_index("tier").loc[TIERS]
rt = read("reach_by_type_tier3").set_index("tier").loc[TIERS]
fig, (a1, a2) = plt.subplots(1, 2, figsize=(10.4, 4.6), gridspec_kw=dict(width_ratios=[1, 1.35], wspace=0.28))
for i, tier in enumerate(TIERS):
    v = r3.loc[tier, "reach_pct"]
    a1.bar(i, v, width=0.6, color=tint(VIOLET, 0.35 + 0.65 * v / r3["reach_pct"].max()), edgecolor=VIOLET, linewidth=1.4)
    a1.text(i, v + 1.5, f"{v:.1f}%", ha="center", fontsize=12, fontweight="bold", color=INK)
    a1.text(i, 3, f"{int(r3.loc[tier, 'n_recipients']):,} of\n{int(r3.loc[tier, 'n_households']):,}", ha="center", va="bottom", fontsize=8.5,
            color="white" if v > 40 else INK2)
a1.set_xticks(range(3)); a1.set_xticklabels([TIER_LABEL[t] for t in TIERS])
a1.set_ylim(0, 108); a1.set_ylabel("Households receiving at least one campaign (%)")
a1.set_title("Any campaign type", loc="left", fontsize=10.5, color=INK2, pad=8); a1.grid(axis="x", visible=False)
w = 0.26
for k, t in enumerate(["TypeA", "TypeB", "TypeC"]):
    xs = np.arange(3) + (k - 1) * w
    vals = rt[t].values
    a2.bar(xs, vals, width=w - 0.02, color=TYPE_COLOR[t] if t == "TypeC" else tint(TYPE_COLOR[t], 0.45),
           label=t.replace("Type", "Type "))
    for x_, v_ in zip(xs, vals):
        a2.text(x_, v_ + 1.2, f"{v_:.1f}", ha="center", fontsize=8.2, color=INK2)
a2.set_xticks(range(3)); a2.set_xticklabels([TIER_LABEL[t] for t in TIERS])
a2.set_ylim(0, 108); a2.set_ylabel("Households receiving the type (%)")
a2.set_title("By campaign type", loc="left", fontsize=10.5, color=INK2, pad=8); a2.grid(axis="x", visible=False)
a2.legend(frameon=False, loc="upper left", fontsize=9)
head(fig, "Campaign reach rose steeply with baseline spending",
     "Share of households in each pre-campaign spending tier that received a campaign during weeks 33\u2013101", top=0.78)
save(fig, "reach-rate")

saved images/reach-rate.png
